In [ ]:
# 2026.03.22 최종 df_sen 버전
sen_out_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv"    

# 2026.03.22_23:45:57 컬럼명 변경

In [ ]:
import pandas as pd
import re
from typing import Iterable, List, Optional, Tuple
def _quote_end_offset(text: str, quote_positions, close_chars):
    """
    Returns:
      0   : ends exactly with a closing quote
     -n   : n tokens after the last closing quote
      NA  : no closing quote in sentence
    """
    if not quote_positions:
        return pd.NA

    # 마지막 닫는 따옴표 찾기
    last_close_pos = None
    for pos, ch in reversed(quote_positions):
        if ch in close_chars:
            last_close_pos = pos
            break

    if last_close_pos is None:
        return pd.NA

    after = text[last_close_pos + 1:].strip()
    if not after:
        return 0

    # 어절 기준
    token_count = len(after.split())
    return -token_count
#---

def annotate_quotes_for_review_sentence_level(
    df: pd.DataFrame,
    text_col: str = "sentence_form",
    id_col: str = "sen_id",
    file_col: str = "file_name",
    order_col: str = "sen_num2",        # or "sen_num"
    quote_chars: Iterable[str] = ('"', '“', '”'),  # straight + smart quotes
    max_quote_span: int = 5,
    add_review_cols: bool = True,
) -> pd.DataFrame:
    """
    Sentence-level quote annotator.

    Adds/updates:
      - quote_count, quote_positions, quote_type
      - quote_group_id, quote_span_len, long_quote
      - quote_start_sen_id, quote_end_sen_id
      - span_wrap_ok, span_type
      - passage_type (IMPORTANT): sentence-level classification
          quote      : sentence is fully quoted (entire sentence inside quote context)
          mixed      : sentence contains narration outside quotes
          narration  : no quote context at all for this sentence
      - review_flag/review_note/quote_type_fix (optional)

    Sentence-level passage_type logic:
      - If sentence is in quote context and has NO narration outside quotes -> quote
      - If sentence has narration before opening quote or after closing quote -> mixed
      - If sentence not in quote context and has no quotes -> narration
    """

    # --- required columns ---
    for c in [text_col, id_col, file_col, order_col]:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    df = df.copy()

    # --- normalize order_col (float -> Int64 safe) ---
    df[order_col] = pd.to_numeric(df[order_col], errors="coerce").astype("Int64")

    # --- stable sort ---
    df = df.sort_values([file_col, order_col], kind="mergesort", na_position="last").reset_index(drop=True)

    quote_chars = list(quote_chars)
    open_chars = ['"', '“']
    close_chars = ['"', '”']

    # ---------- helpers ----------
    def _positions_with_char(s: str) -> List[Tuple[int, str]]:
        out = []
        for ch in quote_chars:
            for m in re.finditer(re.escape(ch), s):
                out.append((m.start(), ch))
        out.sort(key=lambda x: x[0])
        return out

    def _starts_with_open(s: str) -> bool:
        return len(s) > 0 and s[0] in open_chars

    def _ends_with_close(s: str) -> bool:
        return len(s) > 0 and s[-1] in close_chars

    def _has_nonspace(s: str) -> bool:
        return bool(s.strip())

    # Determine if this sentence has narration outside quotes, given current quote context.
    # in_quote_at_start: whether we enter sentence already inside a quote span.
    # positions: sorted list of quote positions (any quote chars)
    # returns (is_mixed, in_quote_at_end)
    def _sentence_mixed_and_endstate(
        text: str,
        positions: List[Tuple[int, str]],
        in_quote_at_start: bool,
    ) -> Tuple[bool, bool]:
        qcount = len(positions)
        in_quote_end = in_quote_at_start ^ (qcount % 2 == 1)  # parity toggle

        # No quotes in this sentence
        if qcount == 0:
            # If we're inside quote context, whole sentence is quote, not mixed
            return (False, in_quote_end)

        # Identify first and last quote positions
        first_q = positions[0][0]
        last_q = positions[-1][0]

        # Narration BEFORE quote starts in this sentence:
        # Only relevant if we are NOT already inside quote at sentence start.
        pre_text = text[:first_q]
        has_pre_narr = (not in_quote_at_start) and _has_nonspace(pre_text)

        # Narration AFTER quote ends in this sentence:
        # This is relevant if the quote gets closed within this sentence.
        # When in_quote_at_start is True and qcount is odd -> closes in this sentence
        # When in_quote_at_start is False and qcount is even -> opens+closes within sentence
        closes_in_this_sentence = (in_quote_at_start and (qcount % 2 == 1)) or ((not in_quote_at_start) and (qcount % 2 == 0))
        post_text = text[last_q + 1:]
        has_post_narr = closes_in_this_sentence and _has_nonspace(post_text)

        is_mixed = has_pre_narr or has_post_narr
        return (is_mixed, in_quote_end)

    # ---------- output columns ----------
    df["quote_count"] = 0
    df["quote_positions"] = None
    df["quote_type"] = None
    df["quote_end_offset"] = pd.NA   # <-- 추가

    df["quote_group_id"] = pd.NA
    df["quote_span_len"] = pd.NA
    df["long_quote"] = False
    df["quote_start_sen_id"] = pd.NA
    df["quote_end_sen_id"] = pd.NA

    df["span_wrap_ok"] = pd.NA
    df["span_type"] = pd.NA  # full_quote_span / partial_quote_span

    # sentence-level
    df["passage_type"] = "narration"  # quote / mixed / narration

    if add_review_cols:
        if "review_flag" not in df.columns:
            df["review_flag"] = False
        if "review_note" not in df.columns:
            df["review_note"] = ""
        if "quote_type_fix" not in df.columns:
            df["quote_type_fix"] = pd.NA


    # ---------- per-file processing ----------
    for file_name, idxs in df.groupby(file_col, sort=False).groups.items():
        idxs = list(idxs)

        # span tracking (for quote_group_id etc.)
        quote_open = False          # "span open" state
        group_id = 0
        span_start_pos: Optional[int] = None
        span_start_sen_id: Optional[str] = None

        # sentence-level quote context (same as quote_open, but we keep explicit)
        in_quote = False

        for pos, row_i in enumerate(idxs):
            sent_id = df.at[row_i, id_col]
            raw = df.at[row_i, text_col]
            text = "" if pd.isna(raw) else str(raw).strip()

            pos_chars = _positions_with_char(text)
            quote_count = len(pos_chars)

            df.at[row_i, "quote_count"] = quote_count
            df.at[row_i, "quote_positions"] = str(pos_chars)
            df.at[row_i, "quote_end_offset"] = _quote_end_offset(text, pos_chars, close_chars)  # <-- 추가

            # ----- sentence-level passage_type -----
            is_mixed, in_quote_end = _sentence_mixed_and_endstate(text, pos_chars, in_quote_at_start=in_quote)

            if (in_quote or quote_count > 0):
                # Sentence participates in quote context somehow
                df.at[row_i, "passage_type"] = "mixed" if is_mixed else "quote"
            else:
                df.at[row_i, "passage_type"] = "narration"

            # update quote context for next sentence
            in_quote = in_quote_end

            # ----- quote_type + span metadata (same as before, parity-based) -----
            if quote_open:
                # inside a multi-sentence span
                if quote_count % 2 == 1:
                    # closes here
                    df.at[row_i, "quote_type"] = "quote_close"
                    df.at[row_i, "quote_group_id"] = group_id

                    span_end_pos = pos
                    span_end_sen_id = sent_id
                    span_len = span_end_pos - span_start_pos + 1  # type: ignore

                    span_row_idxs = [idxs[p] for p in range(span_start_pos, span_end_pos + 1)]  # type: ignore
                    df.loc[span_row_idxs, "quote_span_len"] = span_len
                    df.loc[span_row_idxs, "quote_start_sen_id"] = span_start_sen_id
                    df.loc[span_row_idxs, "quote_end_sen_id"] = span_end_sen_id

                    # span wrap check
                    start_row_i = idxs[span_start_pos]  # type: ignore
                    start_text = "" if pd.isna(df.at[start_row_i, text_col]) else str(df.at[start_row_i, text_col]).strip()
                    end_text = text
                    wrap_ok = _starts_with_open(start_text) and _ends_with_close(end_text)

                    df.loc[span_row_idxs, "span_wrap_ok"] = wrap_ok
                    df.loc[span_row_idxs, "span_type"] = "full_quote_span" if wrap_ok else "partial_quote_span"

                    if span_len > max_quote_span:
                        df.loc[span_row_idxs, "long_quote"] = True
                        if add_review_cols:
                            df.loc[span_row_idxs, "review_flag"] = True
                            df.loc[span_row_idxs, "review_note"] = f"span_len={span_len} (> {max_quote_span})"
                        print(
                            f"⚠️ ({file_name}) quote > {max_quote_span} sentences: "
                            f"start={span_start_sen_id}, end={span_end_sen_id}, len={span_len}"
                        )

                    # reset span state
                    quote_open = False
                    span_start_pos = None
                    span_start_sen_id = None
                else:
                    df.at[row_i, "quote_type"] = "quote_middle"
                    df.at[row_i, "quote_group_id"] = group_id

            else:
                # not in a span
                if quote_count == 0:
                    df.at[row_i, "quote_type"] = "no_quote"
                elif quote_count == 2 and _starts_with_open(text) and _ends_with_close(text):
                    df.at[row_i, "quote_type"] = "full_quote"
                elif quote_count % 2 == 0:
                    df.at[row_i, "quote_type"] = "partial_quote"
                else:
                    df.at[row_i, "quote_type"] = "quote_open"
                    group_id += 1
                    df.at[row_i, "quote_group_id"] = group_id
                    quote_open = True
                    span_start_pos = pos
                    span_start_sen_id = sent_id

        # file ends but span still open
        if quote_open and span_start_pos is not None:
            span_row_idxs = [idxs[p] for p in range(span_start_pos, len(idxs))]
            df.loc[span_row_idxs, "quote_span_len"] = len(idxs) - span_start_pos
            df.loc[span_row_idxs, "quote_start_sen_id"] = span_start_sen_id
            df.loc[span_row_idxs, "quote_end_sen_id"] = pd.NA
            df.loc[span_row_idxs, "span_wrap_ok"] = False
            df.loc[span_row_idxs, "span_type"] = "partial_quote_span"

            if add_review_cols:
                df.loc[span_row_idxs, "review_flag"] = True
                df.loc[span_row_idxs, "review_note"] = "Unclosed quote span at file end"
            print(f"⚠️ ({file_name}) Unclosed quote span at file end: start={span_start_sen_id}")


    return df


# -----------------------------
# Example usage
# -----------------------------
# out_df = annotate_quotes_for_review_sentence_level(df_sen, order_col="sen_num2", max_quote_span=5)
# out_df.to_csv("annotated.csv", index=False, encoding="utf-8")


In [ ]:
#문서 번호 만들기 함수
import numpy as np
import pandas as pd
# 문서 내 'head' 발화 기준으로 docu_no 부여
def make_docu_no(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # -----------------------------
    # 1) docu_id → file_id
    if "file_id" not in df.columns and "docu_id" in df.columns:
        df = df.rename(columns={"docu_id": "file_id"})

    if "file_id" not in df.columns:
        raise KeyError("file_id 또는 docu_id가 필요해.")
    # -----------------------------
    # 2) file_id, sen_id 기준 정렬
    df = df.sort_values(["file_id", "sen_id"], kind="mergesort")

    # 3) head 시작점만 잡기
    sp = df["speaker"].astype("string").str.strip().str.lower()
    is_head = sp.eq("head")

    prev_is_head = is_head.groupby(df["file_id"]).shift(1, fill_value=False)
    head_start = is_head & (~prev_is_head)

    # 3) file별 누적합 → docu_no
    df["docu_no"] = head_start.groupby(df["file_id"]).cumsum().astype("Int64")

    return df

#docu_id만들기 : file_id + docu_no

def add_docu_id_padded_keep_file(df: pd.DataFrame, *, width: int = 3, sep: str = ".") -> pd.DataFrame:
    df = df.copy()

    docu = pd.to_numeric(df["docu_no"], errors="coerce").astype("Int64")
    docu_str = docu.astype(str).str.zfill(width)

    df["docu_id"] = np.where(
        docu.isna(),
        df["file_id"].astype(str),  # docu_no 없으면 file_id만
        df["file_id"].astype(str) + sep + docu_str
    )
    return df



# =========================
# 사용 예시
# =========================
# df = make_docu_no(df_sen)
# df = add_docu_id_padded_keep_file(df, width=3, sep=".")


In [2]:
import pandas as pd
from typing import Optional, Dict
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, only_korean, has_value, FilterValue

In [3]:
SEN_CSV = r"..\csv\sentence\_patched\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.4_sen__patched__2026-02-08_21-09.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
print(f"{SEN_CSV}를 읽어왔습니다.")

df_sen["sen_num2"] = pd.to_numeric(df_sen["sen_num2"], errors="coerce").astype("Int64")
df_sen = df_sen.sort_values(
    ["file_name", "sen_num2"],
    kind="mergesort",
    na_position="last"
)


..\csv\sentence\_patched\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.4_sen__patched__2026-02-08_21-09.csv를 읽어왔습니다.


In [5]:
df_sen.columns

Index(['Unnamed: 0', 'sen_id', 'Column1', 'file_name', 'sen_num2',
       'sentence_form', 'speaker', 'sen_len', 'docu_id', 'sen_num',
       'quote_count', 'quote_positions', 'quote_type', 'quote_group_id',
       'quote_span_len', 'long_quote', 'quote_start_sen_id',
       'quote_end_sen_id', 'span_wrap_ok', 'span_type', 'passage_type',
       'review_flag', 'review_note', 'quote_type_fix', 'modified_at',
       'quote_end_offset'],
      dtype='object')

In [6]:
SEN_ID = "HO0440.00809"

df_sen.loc[df_sen["sen_id"] == SEN_ID, ["sen_id", "sentence_form"]]


,sen_id,sentence_form
1035681,HO0440.00809,"“수의 책이 됨은 위로는 주나라의 {주비산경(周비算經)}으로부터"""


In [ ]:
# 사용 예
output_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.4_sen.csv"
out_df = annotate_quotes_for_review_sentence_level(df_sen,text_col ="sentence_form")
#review_flag 추가
out_df["review_flag"] = (
    (out_df["quote_type"] == "quote_close")
    & (out_df["long_quote"] == True)
    & (out_df["quote_end_offset"].notna())
    & (out_df["quote_end_offset"] != 0)
)


⚠️ (BTAA0009.txt) quote > 5 sentences: start=AA0009.00082, end=AA0009.00089, len=8
⚠️ (BTAA0009.txt) quote > 5 sentences: start=AA0009.00340, end=AA0009.00346, len=7
⚠️ (BTAA0010.txt) quote > 5 sentences: start=AA0010.00890, end=AA0010.00896, len=7
⚠️ (BTAA0013.txt) quote > 5 sentences: start=AA0013.01307, end=AA0013.01314, len=8
⚠️ (BTAA0013.txt) quote > 5 sentences: start=AA0013.03332, end=AA0013.03337, len=6
⚠️ (BTAA0013.txt) quote > 5 sentences: start=AA0013.03342, end=AA0013.03347, len=6
⚠️ (BTAA0013.txt) quote > 5 sentences: start=AA0013.03360, end=AA0013.03365, len=6
⚠️ (BTAA0013.txt) quote > 5 sentences: start=AA0013.03376, end=AA0013.03381, len=6
⚠️ (BTAA0155.txt) quote > 5 sentences: start=AA0155.00860, end=AA0155.00865, len=6
⚠️ (BTAA0155.txt) quote > 5 sentences: start=AA0155.00878, end=AA0155.00883, len=6
⚠️ (BTAA0155.txt) quote > 5 sentences: start=AA0155.01900, end=AA0155.01905, len=6
⚠️ (BTAA0155.txt) quote > 5 sentences: start=AA0155.02265, end=AA0155.02271, len=7
⚠️ (

In [11]:
out_df.columns

Index(['Unnamed: 0', 'sen_id', 'Column1', 'file_name', 'sen_num2',
       'sentence_form', 'speaker', 'sen_len', 'file_id', 'sen_num',
       'quote_count', 'quote_positions', 'quote_type', 'quote_group_id',
       'quote_span_len', 'long_quote', 'quote_start_sen_id',
       'quote_end_sen_id', 'span_wrap_ok', 'span_type', 'passage_type',
       'review_flag', 'review_note', 'quote_type_fix', 'modified_at',
       'quote_end_offset', 'docu_no'],
      dtype='object')

In [ ]:
df_sen.loc[df_sen["file_id"] == "GO0097", "category"] = "상상-아동"
df_sen.loc[df_sen["file_id"] == "BZ0074", "category"] = "총류"

In [ ]:
out_df.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")

In [ ]:
# =========================
# 문서별 문장 번호 만들기 sen_num, rev_sen_num
# =========================
import pandas as pd
from datetime import datetime

# -----------------------------
# 문장 파일 읽기
# -----------------------------
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.6_sen.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
print(f"***sentence 파일 읽기 완료: {SEN_CSV} ({len(df_sen):,}행): {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')} ***")


# -----------------------------------
# 문서별 문장 번호 만들기
# -----------------------------------
# sen_id에서 file 기준 순서 추출
df_sen["file_sen_num"] = (
    df_sen["sen_id"].astype(str).str.split(".").str[-1].astype(int)
)

# 문장 번호
df_sen = df_sen.sort_values(["file_id", "file_sen_num"]).copy()

# docu_id 내부 앞에서부터 순번
df_sen["sen_num"] = (
    df_sen.groupby("docu_id").cumcount() + 1
)

# docu_id 내부 뒤에서부터 순번
df_sen["rev_sen_num"] = (
    df_sen.groupby("docu_id")["sen_num"]
    .transform(lambda x: x.max() - x + 1)
)

print(f"df_sen 생성 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_sen: {df_sen.columns.tolist()}")

In [ ]:
# -----------------------------------
# 문장 끝 시제 대표값 추출
# -----------------------------------
sen_end_tense = (
    df.loc[df["sent_end_V"] == True, ["docu_id", "sen_id", "f_EP_form"]]
    .dropna(subset=["docu_id", "sen_id"])
    .drop_duplicates(subset=["docu_id", "sen_id"], keep="first")
    .rename(columns={"f_EP_form": "sentence_f_EP_form"})
)

# 기존 컬럼 제거 후 merge
for c in ["sentence_f_EP_form", "presentence_f_EP_form", "postsentence_f_EP_form"]:
    if c in df_sen.columns:
        df_sen = df_sen.drop(columns=c)

df_sen = df_sen.merge(
    sen_end_tense,
    on=["docu_id", "sen_id"],
    how="left"
)

In [ ]:
# 문장 파일 저장 컬럼 순서 재배치
from datetime import datetime

front_cols = ['file_id', 'docu_id', 'sen_id', 'file_sen_num', 'sen_num',
       'rev_sen_num']
last_cols = ['quote_count', 'quote_positions', 'quote_group_id', 'quote_span_len', 'long_quote', 'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok', 'span_type', 'passage_type', 'review_flag', 'review_note', 'quote_type_fix', 'modified_at', 'quote_end_offset']
rest_cols = [col for col in df_sen.columns if col not in front_cols + last_cols]
df_sen = df_sen[front_cols + rest_cols + last_cols]


sen_out_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv"    
df_sen.to_csv(sen_out_path, index=False, encoding= "utf-8")
print(f"{sen_out_path}로 저장했습니다.")
print(f"완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

In [ ]:
    # 합계 컬럼명 변경
df_sen = df_sen.rename(columns={
        "last_EN_No": "sentence_f_EN_No",
        "last_EN_label": "sentence_f_EN_label",
        "last_EN_in_quote": "sentence_f_EN_in_quote",
    })

#..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv로 저장했습니다.
# 완료 - 2026.03.22_23:45:57

In [ ]:
# 저장하기
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

output_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv"     

df_sen.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
print(f"완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")